# Phase 3: Machine Learning Modeling

## Healthcare AI Syatem

*Input:* `outputs/model_table.csv` - 25,000 rows
*Output:*
- `models/risk_model.joblib`
- `models/claim_model.joblib`
- `outputs/feature_schema.json`

*Two Models:*
- *Model A* - Visit Risk Classification (Low / Medium / High)
- *Model B* - Claim Outcome Prediction (Paid / Pending / Rejected)

In [2]:
# imports
import pandas as pd
import numpy as np
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score, f1_score)
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

print("Imports done")

Imports done


## Model A: Visit Risk Classification

*Business Purpose:*
Predict whether a hospital visit represents Low, Medium, or High operational and clinical risk.

*Target:* `risk_score` - Low / Medium / High
*Split:* Time-based - earliest 80% train, latest 20% test
*Why time-based?* Prevents data leakage from future visits influencing predictions on past visits.


In [3]:
# Load the dataset
df = pd.read_csv('../outputs/model_table.csv', parse_dates=['registration_date', 'visit_date', 'billing_date'])
print("Shape: ", df.shape)
df.head()

Shape:  (25000, 30)


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,...,risk_numeric,claim_numeric,is_rejected,days_since_registration,visit_frequency,avg_los_per_patient,provider_rejection_rate,visit_month,visit_dayofweek,high_cost_visit_flag
0,2,15,F,Mumbai,CareOne,0,2025-12-27,8,2026-01-01,General,...,0,2,1,5,4,21.120000,0.256876,1,3,0
1,12,3,M,Bangalore,CareOne,0,2025-08-13,65,2026-01-01,ICU,...,2,2,1,141,8,23.750000,0.256876,1,3,1
2,129,44,M,Pune,MediCareX,1,2025-07-20,651,2026-01-01,ICU,...,2,1,0,165,3,32.460000,0.242556,1,3,1
3,133,47,F,Delhi,CareOne,1,2025-11-02,670,2026-01-01,General,...,1,0,0,60,3,30.056667,0.256876,1,3,0
4,139,14,F,Chennai,SecureLife,1,2025-02-05,706,2026-01-01,Cardiology,...,1,0,0,330,9,29.030000,0.157496,1,3,1


In [4]:
# Risk Features
risk_target = 'risk_score'

risk_features = [
    "age",
    "gender",
    "city",
    "insurance_provider",
    "chronic_flag",
    "department",
    "visit_type",
    "doctor_id",
    "length_of_stay_hours",
    "days_since_registration",
    "visit_frequency",
    "avg_los_per_patient",
    "visit_month",
    "visit_dayofweek"
]
risk_df = df[risk_features + [risk_target, "visit_date"]].copy()
risk_df = risk_df.dropna(subset=[risk_target, "visit_date"])
print("Risk dataset shape: ", risk_df.shape)
risk_df.head()

Risk dataset shape:  (25000, 16)


,age,gender,city,insurance_provider,chronic_flag,department,visit_type,doctor_id,length_of_stay_hours,days_since_registration,visit_frequency,avg_los_per_patient,visit_month,visit_dayofweek,risk_score,visit_date
0,15,F,Mumbai,CareOne,0,General,OPD,105,9.63,5,4,21.120000,1,3,Low,2026-01-01
1,3,M,Bangalore,CareOne,0,ICU,ICU,112,59.60,141,8,23.750000,1,3,High,2026-01-01
2,44,M,Pune,MediCareX,1,ICU,ER,150,59.28,165,3,32.460000,1,3,High,2026-01-01
3,47,F,Delhi,CareOne,1,General,OPD,145,25.15,60,3,30.056667,1,3,Medium,2026-01-01
4,14,F,Chennai,SecureLife,1,Cardiology,ER,148,42.88,330,9,29.030000,1,3,Medium,2026-01-01


In [5]:
# Step 3) Time-Based Split
risk_df = risk_df.sort_values("visit_date").reset_index(drop=True)
split_idx = int(len(risk_df) * 0.8)

risk_train = risk_df.iloc[:split_idx].copy()
risk_test = risk_df.iloc[split_idx:].copy()

X_train_risk = risk_train[risk_features]
X_test_risk = risk_test[risk_features]

y_train_risk = risk_train[risk_target]
y_test_risk = risk_test[risk_target]

print("Train shape: ", X_train_risk.shape)
print("Test shape: ", X_test_risk.shape)
print("Train Period: ", risk_train["visit_date"].min().date(), "->", risk_train["visit_date"].max().date())
print("Test Period: ", risk_test["visit_date"].min().date(), "->", risk_test["visit_date"].max().date())

Train shape:  (20000, 14)
Test shape:  (5000, 14)
Train Period:  2025-01-21 -> 2026-01-02
Test Period:  2026-01-02 -> 2026-01-20


## Preprocessing Pipeline

Two Transformers:
- *Numeric* -> SimpleImputer (median strategy)
- *Categorical* -> SimpleImputer + OneHotEncoder

Combined using ColumnTransformer. This entire pipeline gets saved with the model - no separate preprocessing step at inference time.

In [6]:
# Step 5 Preprocessing Pipeline
risk_numeric_features = [
    "age", "chronic_flag", "length_of_stay_hours",
    "days_since_registration", "visit_frequency",
    "doctor_id", "avg_los_per_patient",
    "visit_month", "visit_dayofweek"
]

risk_categorical_features = [
    "gender", "city", "insurance_provider",
    "department", "visit_type"
]

risk_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])
risk_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

risk_preprocessor = ColumnTransformer(transformers=[
    ("numeric", risk_numeric_transformer, risk_numeric_features),
    ("categorical", risk_categorical_transformer, risk_categorical_features)
])

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


## Baseline Model: Logistic Regression

We always start with the simplest model. Logistic Regression gives us a baseline to beat. `class_weight='balanced'` - handles class imbalance as proven in Phase 2 EDA.

In [7]:
# Step 5 Logistic Regression Model
risk_baseline_model = Pipeline(steps=[
    ("preprocessor", risk_preprocessor),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=1000))
])

# Train
risk_baseline_model.fit(X_train_risk, y_train_risk)
print("Baseline model trained.")

# Predict
risk_baseline_predictions = risk_baseline_model.predict(X_train_risk)
risk_baseline_predictions_test = risk_baseline_model.predict(X_test_risk)
print("Baseline model predictions made.")

baseline_acc = accuracy_score(y_test_risk, risk_baseline_predictions_test)
print(f"Baseline model accuracy: {baseline_acc:.4f}")
print("Risk Baseline train accuracy")
print(f"Baseline model train accuracy: {round(accuracy_score(y_train_risk, risk_baseline_predictions), 4):.4f}")
print(f"Baseline model test accuracy: {round(accuracy_score(y_test_risk, risk_baseline_predictions_test), 4):.4f}")
print(f"Risk baseline test weight F1: {round(f1_score(y_test_risk, risk_baseline_predictions_test, average='weighted'), 4):.4f}")

print("\nClassification Report:\n")
print(classification_report(y_test_risk, risk_baseline_predictions_test))
print(f"Confusion Matrix:\n{confusion_matrix(y_test_risk, risk_baseline_predictions_test)}")

Baseline model trained.
Baseline model predictions made.
Baseline model accuracy: 0.9208
Risk Baseline train accuracy
Baseline model train accuracy: 0.9218
Baseline model test accuracy: 0.9208
Risk baseline test weight F1: 0.9211

Classification Report:

              precision    recall  f1-score   support

        High       0.90      0.95      0.93       838
         Low       0.96      0.93      0.94      2360
      Medium       0.88      0.90      0.89      1802

    accuracy                           0.92      5000
   macro avg       0.91      0.93      0.92      5000
weighted avg       0.92      0.92      0.92      5000

Confusion Matrix:
[[ 796    0   42]
 [   0 2186  174]
 [  86   94 1622]]


## Advanced Model: Random Forest

Random Forest captures *non-linear interactions* across patient, department, and visit characteristics.

Logistic Regression assumes linear relationships. Hospital risk is rarely linear - a 70-year-old chronic ICU patient is not just the sum of those individual risk factors.

Key paramters:
- `class_weight='balanced_subsample'` - balances per tree
- `max_depth=8` - prevents overfitting
- `min_samples_leaf=10` - minimum samples at leaf node

Example:
- Doctor 1 -> High
- Doctor 2 -> High
- Doctor 3 -> Medium
- Doctor 4 -> High
- ... Final vote -> High

In [8]:
risk_rf_model = Pipeline(steps=[
    ("preprocessor", risk_preprocessor),
    ("classifier", RandomForestClassifier(
        class_weight='balanced_subsample',
        max_depth=8,
        min_samples_leaf=10,
        n_estimators=200,
        min_samples_split=20,
        random_state=42
    ))
])

# Train
risk_rf_model.fit(X_train_risk, y_train_risk)
print("RF model trained.")

# Predict
risk_rf_predictions = risk_rf_model.predict(X_train_risk)
risk_rf_predictions_test = risk_rf_model.predict(X_test_risk)
print("RF model predictions made.")

rf_acc = accuracy_score(y_test_risk, risk_rf_predictions_test)
print(f"RF model accuracy: {rf_acc:.4f}")
print("Risk RF train accuracy")
print(f"RF model train accuracy: {round(accuracy_score(y_train_risk, risk_rf_predictions), 4):.4f}")
print(f"RF model test accuracy: {round(accuracy_score(y_test_risk, risk_rf_predictions_test), 4):.4f}")
print(f"Risk RF test weight F1: {round(f1_score(y_test_risk, risk_rf_predictions_test, average='weighted'), 4):.4f}")

print("\nRF Classification Report:\n")
print(classification_report(y_test_risk, risk_rf_predictions_test))
print(f"RF Confusion Matrix:\n{confusion_matrix(y_test_risk, risk_rf_predictions_test)}")

RF model trained.
RF model predictions made.
RF model accuracy: 0.9350
Risk RF train accuracy
RF model train accuracy: 0.9414
RF model test accuracy: 0.9350
Risk RF test weight F1: 0.9350

RF Classification Report:

              precision    recall  f1-score   support

        High       0.95      0.93      0.94       838
         Low       0.95      0.95      0.95      2360
      Medium       0.91      0.91      0.91      1802

    accuracy                           0.94      5000
   macro avg       0.94      0.93      0.93      5000
weighted avg       0.94      0.94      0.94      5000

RF Confusion Matrix:
[[ 779    0   59]
 [   0 2253  107]
 [  44  115 1643]]


## XGBoost Model

XGBoost uses *gradient boosting* - builds trees sequentially, each correcting the errors of the previous one.

Why XGBoost here:
- Handles mixed numeric + categorical features well
- Build-in regularisation prevents overfitting
- Typically outperforms Random Forest on structured tabular data

*Note:* XGBoost requires numeric labels. WE use LabelEncoder to convert Low/Medium/High -> 0/1/2. We keep separate variables (y_train_risk_xgb/y_test_risk_xgb) so LR and RF evaluations above are not affected.

In [9]:
# Encode target for CGBoost only
le = LabelEncoder()
risk_df["risk_score_encoded"] = le.fit_transform(risk_df["risk_score"])
print("Label Mapping: ", dict(zip(le.classes_, le.transform(le.classes_))))

Label Mapping:  {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}


In [10]:
# Time-based split for XGBoost
# Using same split_idx - consistent with LR and RF
risk_df = risk_df.sort_values("visit_date").reset_index(drop=True)
split_idx = int(len(risk_df) * 0.8)

risk_train_xgb = risk_df.iloc[:split_idx].copy()
risk_test_xgb = risk_df.iloc[split_idx:].copy()

X_train_risk = risk_train_xgb[risk_features]
X_test_risk = risk_test_xgb[risk_features]

# Separate XGB label variables - does NOT overwrite y_train_risk
y_train_risk_xgb = risk_train_xgb["risk_score_encoded"]
y_test_risk_xgb = risk_test_xgb["risk_score_encoded"]

print("XGBoost train shape:", X_train_risk.shape)
print("XGBoost test shape:", X_test_risk.shape)

XGBoost train shape: (20000, 14)
XGBoost test shape: (5000, 14)


In [11]:
# Step 7 XGBoost Model
risk_xgb_model = Pipeline(steps=[
    ("preprocessor", risk_preprocessor),
    ("classifier", XGBClassifier(
        objective="multi:softmax",
        num_class=3,
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0
    ))
])

risk_xgb_model.fit(X_train_risk, y_train_risk_xgb)

risk_xgb_pred_train = risk_xgb_model.predict(X_train_risk)
risk_xgb_pred_test = risk_xgb_model.predict(X_test_risk)

xgb_acc = accuracy_score(y_test_risk_xgb, risk_xgb_pred_test)

print("Risk XGB Train Accuracy:", round(accuracy_score(y_train_risk_xgb, risk_xgb_pred_train), 4))
print("Risk XGB Test Accuracy:", round(accuracy_score(y_test_risk_xgb, risk_xgb_pred_test), 4))
print("Risk XGB Test Weighted F1 Score:", round(f1_score(y_test_risk_xgb, risk_xgb_pred_test, average="weighted"), 4))
print("\nClassification Report:\n")
print(classification_report(y_test_risk_xgb, risk_xgb_pred_test))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test_risk_xgb, risk_xgb_pred_test))

Risk XGB Train Accuracy: 0.9943
Risk XGB Test Accuracy: 0.9544
Risk XGB Test Weighted F1 Score: 0.9545

Classification Report:

              precision    recall  f1-score   support

           0       0.98      0.94      0.96       840
           1       0.97      0.96      0.97      2360
           2       0.92      0.95      0.94      1800

    accuracy                           0.95      5000
   macro avg       0.96      0.95      0.95      5000
weighted avg       0.95      0.95      0.95      5000


Confusion Matrix:

[[ 789    0   51]
 [   0 2272   88]
 [  14   75 1711]]


In [12]:
# Model Comparison
print("=" * 45)
print("MODEL A - RISK CLASSIFICATION COMPARISON")
print("=" * 45)
print(f"Logistic Regression : {baseline_acc:.4f}")
print(f"Random Forest       : {rf_acc:.4f}")
print(f"XGBoost             : {xgb_acc:.4f}")
print()
print(f"Best model: XGBoost -> {max(baseline_acc, rf_acc, xgb_acc):.4f}")

MODEL A - RISK CLASSIFICATION COMPARISON
Logistic Regression : 0.9208
Random Forest       : 0.9350
XGBoost             : 0.9544

Best model: XGBoost -> 0.9544


# Optional – Hyperparameter Tuning: Random Forest – Not needed at the moment

We use randomizedSearchCV to find the best RF parameters. `f1_weighted` is our scoring metric – not accuracy. This ensures minority classes (High risk) are weighted properly.


In [13]:
rf_param_grid = {
    "classifier__n_estimators": [200, 300, 400],
    "classifier__max_depth": [8, 12, 16, None],
    "classifier__min_samples_split": [5, 10, 20],
    "classifier__min_samples_leaf": [1, 2, 4]
}

rf_random_search = RandomizedSearchCV(
    risk_rf_model,                          # the pipeline to tune
    param_distributions=rf_param_grid,      # values to try
    n_iter=10,                              # try only 10 random combinations
    cv=3,                                   # 3-fold cross-validation
    scoring="f1_weighted",                  # optimize for weighted F1 score
    n_jobs=-1,                              # use all CPU cores
    random_state=42                         # reproducible random selection
)

rf_random_search.fit(X_train_risk, y_train_risk)

print("Best Parameters:", rf_random_search.best_params_)

Best Parameters: {'classifier__n_estimators': 200, 'classifier__min_samples_split': 5, 'classifier__min_samples_leaf': 4, 'classifier__max_depth': 12}


In [14]:
best_rf_model = rf_random_search.best_estimator_
rf_tuned_pred = best_rf_model.predict(X_test_risk)

print("Tuned RF Accuracy:", round(accuracy_score(y_test_risk, rf_tuned_pred), 4))
print("Tuned RF Weighted F1 Score:", round(f1_score(y_test_risk, rf_tuned_pred, average="weighted"), 4))
print("\nClassification Report:")
print(classification_report(y_test_risk, rf_tuned_pred))

Tuned RF Accuracy: 0.435
Tuned RF Weighted F1 Score: 0.3815

Classification Report:
              precision    recall  f1-score   support

        High       0.23      0.05      0.08       838
         Low       0.47      0.74      0.57      2360
      Medium       0.36      0.22      0.27      1802

    accuracy                           0.43      5000
   macro avg       0.35      0.33      0.31      5000
weighted avg       0.39      0.43      0.38      5000



Takes 20,000 training rows
Randomly shuffles them
Splits into 3 folds

Fold 1: rows 3, 7, 12, 19, 25 <- random mix of all dates
Fold 2: rows 1, 4, 9, 14, 22 <- random mix of all dates
Fold 3: rows 2, 5, 8, 11, 17 <- random mix of all dates


In [15]:
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=3)
# Train on past -> validate on future - every fold
# Fold 1: train Jan-Apr, validate May
# Fold 2: train Jan-Jul, validate Aug
# Fold 3: train Jan-Oct, validate Nov

## Model B - Claim Outcome Prediction

*Business Purpose*
 Predicts whether an insurance claim will be Paid, Pending, or Rejected - before submission.

*Target:* `claim_status` - Paid/Pending/Rejected
*Split:* Time-based on `billing_date`

*Leakage prevention:*
`approved_amount` and `payment_days` excluded - both are post-outcome variables that are only known AFTER the claim is processed.

#                     Model A                    Model B
#Predicts            risk_score            claim_status
#Split on            visit_date             billing_date
#Excluded cols       billing columns         approved_amount, payment_days
#Business value      early intervention      revenue recovery

In [17]:
# Step 1. Define claim features
claim_target = "claim_status"
claim_features = [
    "age",
    "gender",
    "city",
    "insurance_provider",
    "chronic_flag",
    "department",
    "visit_type",
    "doctor_id",
    "length_of_stay_hours",
    "risk_score",
    "billed_amount",
    "days_since_registration",
    "visit_frequency",
    "avg_los_per_patient",
    "provider_rejection_rate",
    "visit_month",
    "visit_dayofweek",
    "high_cost_visit_flag"
]

claim_df = df[claim_features + [claim_target, "billing_date"]].copy()
claim_df = claim_df.dropna(subset=[claim_target, "billing_date"])
print("Claim Dataset Shape: ", claim_df.shape)
claim_df.head()

Claim Dataset Shape:  (25000, 20)


,age,gender,city,insurance_provider,chronic_flag,department,visit_type,doctor_id,length_of_stay_hours,risk_score,billed_amount,days_since_registration,visit_frequency,avg_los_per_patient,provider_rejection_rate,visit_month,visit_dayofweek,high_cost_visit_flag,claim_status,billing_date
0,15,F,Mumbai,CareOne,0,General,OPD,105,9.63,Low,9612.77,5,4,21.120000,0.256876,1,3,0,Rejected,2026-01-19
1,3,M,Bangalore,CareOne,0,ICU,ICU,112,59.60,High,88539.01,141,8,23.750000,0.256876,1,3,1,Rejected,2026-01-05
2,44,M,Pune,MediCareX,1,ICU,ER,150,59.28,High,88539.01,165,3,32.460000,0.242556,1,3,1,Pending,2026-01-20
3,47,F,Delhi,CareOne,1,General,OPD,145,25.15,Medium,20958.52,60,3,30.056667,0.256876,1,3,0,Paid,2026-01-15
4,14,F,Chennai,SecureLife,1,Cardiology,ER,148,42.88,Medium,74921.48,330,9,29.030000,0.157496,1,3,1,Paid,2026-01-16


In [20]:
# Step 2 claim status distribution
print("Claim Status Distribution:")
print(claim_df[claim_target].value_counts())
print()
print("Claim status % distribution")
print((claim_df[claim_target].value_counts(normalize=True)*100).round(2))

Claim Status Distribution:
claim_status
Paid        13638
Pending      6096
Rejected     5266
Name: count, dtype: int64

Claim status % distribution
claim_status
Paid        54.55
Pending     24.38
Rejected    21.06
Name: proportion, dtype: float64


## Step 3 Time-Based Split for Claim Model

Sorting by `billing_date` - not `visit_date`. Billing events happen after visits. We respect that temporal order.

In [22]:
claim_df = claim_df.sort_values(by="billing_date").reset_index(drop=True)
split_idx = int(len(claim_df)*0.8)

claim_train = claim_df.iloc[:split_idx].copy()
claim_test = claim_df.iloc[split_idx:].copy()

X_train_claim = claim_train[claim_features]
X_test_claim = claim_test[claim_features]

y_train_claim = claim_train[claim_target]
y_test_claim = claim_test[claim_target]

print(f"Train Shape: {X_train_claim.shape}")
print(f"Test Shape: {X_test_claim.shape}")
print(f"Train period: {claim_train['billing_date'].min().date()} - {claim_train['billing_date'].max().date()}")
print(f"Test period: {claim_test['billing_date'].min().date()} - {claim_test['billing_date'].max().date()}")

Train Shape: (20000, 18)
Test Shape: (5000, 18)
Train period: 2025-01-28 - 2026-01-17
Test period: 2026-01-17 - 2026-01-20


## Step 4 Preprocessing Pipeline for Claim Model

Similar to the risk model but a claim model has additional features:
- `risk_score` as a categorical input feature
- `billed_amount` as numeric
- `provider_rejection_rate` as numeric

In [23]:
claim_numeric_features = [
    "age", "chronic_flag", "doctor_id",
    "length_of_stay_hours", "billed_amount",
    "days_since_registration", "visit_frequency",
    "avg_los_per_patient", "provider_rejection_rate",
    "visit_month", "visit_dayofweek", "high_cost_visit_flag"
]

claim_categorical_features = [
    "gender", "city", "insurance_provider",
    "department", "visit_type", "risk_score"
]

claim_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

claim_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

claim_preprocessor = ColumnTransformer(transformers=[
    ("num", claim_numeric_transformer, claim_numeric_features),
    ("cat", claim_categorical_transformer, claim_categorical_features)
])

print("Claim preprocessing pipeline created.")

Claim preprocessing pipeline created.


In [24]:
# Step 5 Logistic Regression Model
claim_baseline_model = Pipeline(steps=[
    ("preprocessor", claim_preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

# Train -> Predict -> Evaluate
# Train
claim_baseline_model.fit(X_train_claim, y_train_claim)

# Predict
claim_baseline_pred_train = claim_baseline_model.predict(X_train_claim)
claim_baseline_pred_test = claim_baseline_model.predict(X_test_claim)

# Evaluate
claim_baseline_acc = accuracy_score(y_test_claim, claim_baseline_pred_test)

print(f"Claim Baseline Train Accuracy: {round(accuracy_score(y_train_claim, claim_baseline_pred_train), 4)}")
print(f"Claim Baseline Test Accuracy: {round(accuracy_score(y_test_claim, claim_baseline_pred_test), 4)}")
print(f"Claim Baseline Test Weighted F1 Score: {round(f1_score(y_test_claim, claim_baseline_pred_test, average='weighted'), 4)}")
print("\nClassification Report:\n")
print(classification_report(y_test_claim, claim_baseline_pred_test))
print(f"Confusion Matrix:\n {confusion_matrix(y_test_claim, claim_baseline_pred_test)}")


Claim Baseline Train Accuracy: 0.4829
Claim Baseline Test Accuracy: 0.4194
Claim Baseline Test Weighted F1 Score: 0.4318

Classification Report:

              precision    recall  f1-score   support

        Paid       0.64      0.50      0.56      2771
     Pending       0.24      0.21      0.22      1189
    Rejected       0.25      0.44      0.32      1040

    accuracy                           0.42      5000
   macro avg       0.38      0.38      0.37      5000
weighted avg       0.46      0.42      0.43      5000

Confusion Matrix:
 [[1395  540  836]
 [ 448  248  493]
 [ 348  238  454]]


In [31]:
# Step 6 Random Forest Model
claim_rf_model = Pipeline(steps=[
    ("preprocessor", claim_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=250,
        max_depth=14,
        min_samples_split=8,
        class_weight="balanced",
        random_state=42
    ))
])

# Train -> Predict -> Evaluate
# Train
claim_rf_model.fit(X_train_claim, y_train_claim)

# Predict
claim_rf_pred_train = claim_rf_model.predict(X_train_claim)
claim_rf_pred_test = claim_rf_model.predict(X_test_claim)

# Evaluate
claim_rf_acc = accuracy_score(y_test_claim, claim_rf_pred_test)

print(f"Claim RF Train Accuracy: {round(accuracy_score(y_train_claim, claim_rf_pred_train), 4)}")
print(f"Claim RF Test Accuracy: {round(accuracy_score(y_test_claim, claim_rf_pred_test), 4)}")
print(f"Claim RF Test Weighted F1 Score: {round(f1_score(y_test_claim, claim_rf_pred_test, average='weighted'), 4)}")
print("\nClassification Report:\n")
print(classification_report(y_test_claim, claim_rf_pred_test))
print(f"Confusion Matrix:\n {confusion_matrix(y_test_claim, claim_rf_pred_test)}")

Claim RF Train Accuracy: 0.9797
Claim RF Test Accuracy: 0.4132
Claim RF Test Weighted F1 Score: 0.4066

Classification Report:

              precision    recall  f1-score   support

        Paid       0.55      0.59      0.57      2771
     Pending       0.23      0.21      0.22      1189
    Rejected       0.19      0.18      0.19      1040

    accuracy                           0.41      5000
   macro avg       0.33      0.33      0.33      5000
weighted avg       0.40      0.41      0.41      5000

Confusion Matrix:
 [[1630  599  542]
 [ 696  248  245]
 [ 628  224  188]]


In [26]:
# XGBoost
# XGBoost needs numeric labels
le_claim = LabelEncoder()
claim_df["claim_status_encoded"] = le_claim.fit_transform(claim_df["claim_status"])
print(f"Label mapping:", dict(zip(le_claim.classes_, le_claim.transform(le_claim.classes_))))

Label mapping: {'Paid': np.int64(0), 'Pending': np.int64(1), 'Rejected': np.int64(2)}


In [27]:
# Time based spliting
claim_df = claim_df.sort_values("billing_date").reset_index(drop=True)
split_idx = int(len(claim_df) * 0.8)

claim_train_xgb = claim_df.iloc[:split_idx].copy()
claim_test_xgb = claim_df.iloc[split_idx:].copy()

X_train_claim = claim_train_xgb[claim_features]
X_test_claim = claim_test_xgb[claim_features]

# Separate XGB label variables - does NOT overwrite y_train_claim
y_train_claim_xgb = claim_train_xgb["claim_status_encoded"]
y_test_claim_xgb = claim_test_xgb["claim_status_encoded"]

print("XGBoost train shape:", X_train_claim.shape)
print("XGBoost test shape:", X_test_claim.shape)

XGBoost train shape: (20000, 18)
XGBoost test shape: (5000, 18)


In [29]:
# Step 7 XGBoost Model
claim_xgb_model = Pipeline(steps=[
    ("preprocessor", claim_preprocessor),
    ("classifier", XGBClassifier(
        objective="multi:softmax",
        num_class=3,
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0
    ))
])

# Train -> Predict -> Evaluate
# Train
claim_xgb_model.fit(X_train_claim, y_train_claim_xgb)

# Predict
claim_xgb_pred_train = claim_xgb_model.predict(X_train_claim)
claim_xgb_pred_test = claim_xgb_model.predict(X_test_claim)

# Evaluate
claim_xgb_acc = accuracy_score(y_test_claim_xgb, claim_xgb_pred_test)

print(f"Claim xgb Train Accuracy: {round(accuracy_score(y_train_claim_xgb, claim_xgb_pred_train), 4)}")
print(f"Claim xgb Test Accuracy: {round(accuracy_score(y_test_claim_xgb, claim_xgb_pred_test), 4)}")
print(f"Claim xgb Test Weighted F1 Score: {round(f1_score(y_test_claim_xgb, claim_xgb_pred_test, average='weighted'), 4)}")
print("\nClassification Report:\n")
print(classification_report(y_test_claim_xgb, claim_xgb_pred_test))
print(f"Confusion Matrix:\n {confusion_matrix(y_test_claim_xgb, claim_xgb_pred_test)}")

Claim xgb Train Accuracy: 0.765
Claim xgb Test Accuracy: 0.5476
Claim xgb Test Weighted F1 Score: 0.474

Classification Report:

              precision    recall  f1-score   support

           0       0.59      0.89      0.71      2773
           1       0.33      0.11      0.17      1188
           2       0.35      0.14      0.20      1039

    accuracy                           0.55      5000
   macro avg       0.42      0.38      0.36      5000
weighted avg       0.48      0.55      0.47      5000

Confusion Matrix:
 [[2457  158  158]
 [ 933  131  124]
 [ 785  104  150]]


In [32]:
print("="*45)
print("MODEL B - CLAIM OUTCOME COMPARISON")
print("="*45)
print(f"Logistic Regression  : {claim_baseline_acc:.4f}")
print(f"Random Forest        : {claim_rf_acc:.4f}")
print(f"XGBoost              : {claim_xgb_acc:.4f}")
print()
models = {
    "Logistic Regression": claim_baseline_acc,
    "Random Forest": claim_rf_acc,
    "XGBoost": claim_xgb_acc
}
best = max(models, key=models.get)
print(f"Best Model: {best} -> {models[best]:.4f}")

### Three Things Different From Model A XGBoost
#le_claim -> Separate LabelEncoder for claim labels
#billing_date -> split column instead of visit_date
#target_names=le_claim.classes_ -> shows Paid/Pending/Rejected not 0/1/2

MODEL B - CLAIM OUTCOME COMPARISON
Logistic Regression  : 0.4998
Random Forest        : 0.4132
XGBoost              : 0.5476

Best Model: XGBoost -> 0.5476


## Save Model Artifacts

We save both final models using joblib. At inference time - FastAPI loads these files at startup. No retraining needed per request.

In [33]:
joblib.dump(risk_rf_model, "../models/risk_model.joblib")
joblib.dump(claim_rf_model, "../models/claim_model.joblib")

print("Models saved:")
print("../models/risk_model.joblib")
print("../models/claim_model.joblib")

Models saved:
../models/risk_model.joblib
../models/claim_model.joblib


## Save Feature Schema

The feature schema tells our FastAPI service:
- Which features to expect in the request payload
- Which column is the target
- What split strategy was used

This is the contract between the model and the API.

In [34]:
feature_schema = {
    "risk_model_features": risk_features, # list of 14 features
    "claim_model_features": claim_features, # list of 18 features
    "risk_target": "risk_score", # what Model A predicts
    "claim_target": "claim_status", # what Model B predicts
    "risk_time_column": "visit_date", # split column for Model A
    "claim_time_column": "billing_date", # split column for Model B
    "split_strategy": "earliest 80 percent train, latest 20 percent test"
}

with open("../outputs/feature_schema.json", "w") as f:
    json.dump(feature_schema, f, indent=4)

print("Feature schema saved to ../outputs/feature_schema.json")

Feature schema saved to ../outputs/feature_schema.json
